# 02 — Composite wetness W + DFAA index + event labels
Builds the forecast target (Eq. 2) and train-fit event thresholds. Writes `artefacts/dfaa.npz`.

In [ ]:
# === Colab/local auto-setup (device + data path) ===
import sys, os, subprocess
from pathlib import Path
def _pip(*pkgs):
    for p in pkgs:
        mod = p.split('==')[0].replace('-', '_').replace('scikit_learn', 'sklearn')
        try:
            __import__(mod)
        except Exception:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])
_pip('numpy', 'pandas', 'scipy', 'scikit-learn', 'statsmodels', 'torch', 'matplotlib')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
CSV = 'Bangladesh Meterological data.csv'
cands = [Path.cwd()/CSV, Path('/content')/CSV, Path(r'd:\BUET RESEARCH WORK\Bangladesh Flood')/CSV]
root = next((c.parent for c in cands if c.exists()), None)
if root is None:
    try:
        from google.colab import files
        files.upload(); root = Path.cwd()
    except Exception:
        raise FileNotFoundError('Upload "%s" next to this notebook.' % CSV)
os.environ['DFAA_ROOT'] = str(root)
print('DFAA_ROOT =', root)


In [ ]:
%%writefile common.py
"""Common config, data loader, and leak-free helpers for the Bangladesh DFAA study.

All paths hardcoded (workspace convention). Train-only fits everywhere.
Notation matches EXPERIMENT_DESIGN.md. No experiment numbers are produced here;
this is the shared, smoke-testable core that the notebooks reuse.
"""
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

# ROOT overridable so the shipped notebooks run on Colab/local (set env DFAA_ROOT).
ROOT = Path(os.environ.get("DFAA_ROOT", r"d:\BUET RESEARCH WORK\Bangladesh Flood"))
RAW_CSV = ROOT / "Bangladesh Meterological data.csv"
ART = ROOT / "artefacts"
ART.mkdir(exist_ok=True)

# ---- locked study constants (EXPERIMENT_DESIGN.md defaults) ----
SEED = 0
VARS_Z = ["Rainfall_mm", "Soil_moisture_mm"]          # standardized by train climatology
# robust standardization: per-(s,m) sd floored at SD_FLOOR_FRAC * station-pooled train sd,
# then z clipped to +-Z_CLIP. Guards the soil-moisture saturation / dry-month near-zero-sd
# pathology (otherwise z -> ~-40000). Fixed/train-only transforms => no leakage; train cells
# (|z|<3.6) are untouched. Documented in RESULTS_LOG S1/S2.
SD_FLOOR_FRAC = 0.15
Z_CLIP = 4.0
W_WEIGHTS = (1 / 3, 1 / 3, 1 / 3)                     # w1*z_P + w2*z_SM + w3*SPEI
ALPHA_DFAA = 1.8                                       # Wu (2006) constant in Eq. 2
LEADS = (1, 2, 3)                                      # symmetric window scale = lead h
TAUS = np.array([0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95], dtype=np.float64)
THETA_PCT = 80.0                                       # theta_D = 80th pct of |DFAA| on train

# time-ordered split by ORIGIN year (the month t at which DFAA(s,t) is anchored)
TRAIN_YEARS = (2000, 2014)
VAL_YEARS = (2015, 2017)
TEST_YEARS = (2018, 2022)

# BMD station latitudes/longitudes (deg) for PET extraterrestrial radiation + maps.
# Standard BMD station coordinates; for PET only latitude matters (Ra is ~flat to +-0.2 deg).
# Provenance flagged for final verification before the .tex (CLAUDE.md Rule 3).
STATION_LATLON = {
    "Barisal":              (22.70, 90.37),
    "Bogra":                (24.85, 89.37),
    "Chittagong(Air-port)": (22.25, 91.81),
    "Comilla":              (23.43, 91.18),
    "Cox's Bazar":          (21.45, 91.97),
    "Dhaka":                (23.78, 90.38),
    "Faridpur":             (23.60, 89.85),
    "Jessore":              (23.18, 89.16),
    "Khulna":               (22.78, 89.53),
    "Mymensingh":           (24.75, 90.43),
    "Rajshahi":             (24.37, 88.70),
    "Rangpur":              (25.73, 89.23),
    "Sylhet":               (24.90, 91.88),
}


def load_clean():
    """Load raw CSV, drop the trailing all-NaN row, sort, add integer month index t.

    Returns a tidy long DataFrame with columns:
      Station_Name, Station_Code, Year, Month, Max_Temp, Min_Temp, Rainfall_mm,
      Soil_moisture_mm, SPEI_3, s (0..12 station id), t (0-based global month index),
      origin_year, split.
    Raw file is never modified.
    """
    df = pd.read_csv(RAW_CSV)
    # drop rows that are entirely NaN in the value columns (the trailing NaN row)
    val_cols = ["Max_Temp", "Min_Temp", "Rainfall_mm", "Soil_moisture_mm", "SPEI_3"]
    before = len(df)
    df = df.dropna(subset=["Station_Name", "Year", "Month"], how="any").copy()
    df = df.dropna(subset=val_cols, how="all").copy()
    dropped = before - len(df)

    df["Year"] = df["Year"].astype(int)
    df["Month"] = df["Month"].astype(int)
    df = df.sort_values(["Station_Name", "Year", "Month"]).reset_index(drop=True)

    stations = sorted(df["Station_Name"].unique())
    sid = {name: i for i, name in enumerate(stations)}
    df["s"] = df["Station_Name"].map(sid)

    # global 0-based month index over 2000-01 .. 2022-12
    df["t"] = (df["Year"] - 2000) * 12 + (df["Month"] - 1)

    def split_of(y):
        if TRAIN_YEARS[0] <= y <= TRAIN_YEARS[1]:
            return "train"
        if VAL_YEARS[0] <= y <= VAL_YEARS[1]:
            return "val"
        return "test"

    df["origin_year"] = df["Year"]
    df["split"] = df["Year"].map(split_of)
    return df, stations, sid, dropped


def to_grid(df, col):
    """Return an (S, T) float array of `col` indexed by [station s, month index t],
    NaN where missing. S=13 stations, T=276 months (2000-01..2022-12)."""
    S = df["s"].nunique()
    T = 276
    g = np.full((S, T), np.nan, dtype=np.float64)
    g[df["s"].to_numpy(), df["t"].to_numpy()] = df[col].to_numpy(dtype=float)
    return g


def train_mask_t(years=TRAIN_YEARS):
    """Boolean length-276 mask of month indices whose calendar year is in `years`."""
    t = np.arange(276)
    yr = 2000 + t // 12
    return (yr >= years[0]) & (yr <= years[1])


def fit_climatology(grid, train_t):
    """Per (station s, calendar month m) mean/std on TRAIN months only, with a robust
    std floor at SD_FLOOR_FRAC * station-pooled train sd (guards saturated/dry near-zero-sd
    cells). grid: (S,T); train_t: bool length T. Returns mu,sd as (S,12)."""
    S, T = grid.shape
    mu = np.full((S, 12), np.nan)
    sd = np.full((S, 12), np.nan)
    months = np.arange(T) % 12
    for s in range(S):
        for m in range(12):
            sel = (months == m) & train_t
            vals = grid[s, sel]
            vals = vals[~np.isnan(vals)]
            if len(vals) >= 2:
                mu[s, m] = vals.mean()
                sd[s, m] = vals.std(ddof=1)
    glob = np.nanstd(grid[:, train_t])
    for s in range(S):
        stat_sd = np.nanstd(grid[s, train_t])
        floor = SD_FLOOR_FRAC * stat_sd if np.isfinite(stat_sd) and stat_sd > 1e-6 else glob
        floor = max(floor, 1e-6)
        for m in range(12):
            if not np.isfinite(sd[s, m]) or sd[s, m] < floor:
                sd[s, m] = floor
            if not np.isfinite(mu[s, m]):
                mu[s, m] = np.nanmean(grid[s, train_t])
    return mu, sd


def standardize(grid, mu, sd):
    """z_v(s,t) = clip( (x - mu[s,m]) / sd[s,m], -Z_CLIP, +Z_CLIP ), m = t%12. NaNs propagate."""
    S, T = grid.shape
    months = np.arange(T) % 12
    z = (grid - mu[:, months]) / sd[:, months]
    return np.clip(z, -Z_CLIP, Z_CLIP)


In [ ]:
"""Step 2 - composite wetness W, DFAA index (Eq. 2), event labels, climatology of events.

Implements EXPERIMENT_DESIGN.md Step 2 with the locked disambiguation:
  * symmetric window scale = lead h in {1,2,3}
  * W_E(s,t,h) = mean W over [t-h+1 .. t]   (known at origin t)
  * W_L(s,t,h) = mean W over [t+1 .. t+h]   (the forecast-unknown future)
  * DFAA(s,t,h) = (W_L - W_E)(|W_E|+|W_L|) * alpha^(-|W_E+W_L|),  alpha=1.8
  * target modeled directly is DFAA(s,t,h)
  * theta_D(h) = 80th pct of |DFAA| over TRAIN origins only
Persists artefacts/dfaa.npz + artefacts/dfaa_meta.json. Not yet a forecast (no skill metric);
this builds the label/target the later experiments predict.
"""
import json
import numpy as np
import pandas as pd
from common import (ROOT, ART, load_clean, to_grid, train_mask_t, fit_climatology,
                    standardize, VARS_Z, W_WEIGHTS, ALPHA_DFAA, LEADS, THETA_PCT,
                    TRAIN_YEARS, SEED)

np.random.seed(SEED)
print("=" * 70)
print("STEP 2 - composite wetness W + DFAA index + event labels")
print("=" * 70)

df, stations, sid, _ = load_clean()
S, T = len(stations), 276
train_t = train_mask_t()

# ---- composite wetness W(s,t) = w1 z_P + w2 z_SM + w3 SPEI (train climatology) ----
zP = standardize(to_grid(df, "Rainfall_mm"), *fit_climatology(to_grid(df, "Rainfall_mm"), train_t))
zSM = standardize(to_grid(df, "Soil_moisture_mm"), *fit_climatology(to_grid(df, "Soil_moisture_mm"), train_t))
SPEI = to_grid(df, "SPEI_3")
w1, w2, w3 = W_WEIGHTS
W = w1 * zP + w2 * zSM + w3 * SPEI                      # (S,T), NaN where any input missing
print(f"\nComposite W: weights {W_WEIGHTS}")
print(f"  W finite cells: {np.isfinite(W).sum()} / {S*T}")
print(f"  W (all):  mean={np.nanmean(W):+.3f}  std={np.nanstd(W):.3f}  "
      f"min={np.nanmin(W):+.2f}  max={np.nanmax(W):+.2f}")
print(f"  W (train) mean={np.nanmean(W[:, train_t]):+.3f}  std={np.nanstd(W[:, train_t]):.3f}")
print(f"  component corr on train: "
      f"corr(zP,SPEI)={np.corrcoef(zP[:,train_t][np.isfinite(zP[:,train_t]*SPEI[:,train_t])], SPEI[:,train_t][np.isfinite(zP[:,train_t]*SPEI[:,train_t])])[0,1]:+.2f}  "
      f"corr(zSM,SPEI)={np.corrcoef(zSM[:,train_t][np.isfinite(zSM[:,train_t]*SPEI[:,train_t])], SPEI[:,train_t][np.isfinite(zSM[:,train_t]*SPEI[:,train_t])])[0,1]:+.2f}")


def window_mean(arr, s, lo, hi):
    """mean of arr[s, lo..hi] inclusive; NaN if out of range or any element missing."""
    if lo < 0 or hi >= T:
        return np.nan
    seg = arr[s, lo:hi + 1]
    if np.any(~np.isfinite(seg)):
        return np.nan
    return float(seg.mean())


# ---- DFAA per lead ----
DFAA = {h: np.full((S, T), np.nan) for h in LEADS}
W_E = {h: np.full((S, T), np.nan) for h in LEADS}
W_L = {h: np.full((S, T), np.nan) for h in LEADS}
yr = 2000 + np.arange(T) // 12

for h in LEADS:
    for s in range(S):
        for t in range(T):
            we = window_mean(W, s, t - h + 1, t)          # early (known)
            wl = window_mean(W, s, t + 1, t + h)           # late (future)
            if not (np.isfinite(we) and np.isfinite(wl)):
                continue
            W_E[h][s, t] = we
            W_L[h][s, t] = wl
            flip = wl - we
            mag = abs(we) + abs(wl)
            supp = ALPHA_DFAA ** (-abs(we + wl))
            DFAA[h][s, t] = flip * mag * supp

# ---- event thresholds theta_D(h) on TRAIN origins only ----
meta = {"weights": list(W_WEIGHTS), "alpha": ALPHA_DFAA, "leads": list(LEADS),
        "theta_pct": THETA_PCT, "theta_D": {}, "stations": stations,
        "window": "symmetric: E=[t-h+1..t], L=[t+1..t+h]"}
print("\nPer-lead DFAA distribution, threshold theta_D, and event counts:")
print(f"{'h':>2} {'n_valid':>8} {'n_train':>8} {'DFAA std':>9} {'|DFAA|max':>10} "
      f"{'theta_D':>8} {'DTF_all':>8} {'FTD_all':>8} {'evt%':>6}")
ev_labels = {}   # h -> (S,T) in {-1,0,+1}; +1=DTF, -1=FTD, 0=none, NaN where no DFAA
for h in LEADS:
    d = DFAA[h]
    valid = np.isfinite(d)
    train_cells = valid & train_t[None, :]
    theta = float(np.percentile(np.abs(d[train_cells]), THETA_PCT))
    meta["theta_D"][str(h)] = theta
    lab = np.full((S, T), np.nan)
    lab[valid] = 0.0
    lab[valid & (d >= theta)] = 1.0     # DTF (drought->flood)
    lab[valid & (d <= -theta)] = -1.0   # FTD (flood->drought)
    ev_labels[h] = lab
    n_dtf = int(np.nansum(lab == 1.0))
    n_ftd = int(np.nansum(lab == -1.0))
    n_valid = int(valid.sum())
    print(f"{h:>2} {n_valid:>8} {int(train_cells.sum()):>8} {np.nanstd(d):>9.3f} "
          f"{np.nanmax(np.abs(d)):>10.3f} {theta:>8.3f} {n_dtf:>8} {n_ftd:>8} "
          f"{100*(n_dtf+n_ftd)/n_valid:>5.1f}%")

# ---- event counts by split (h=2 headline) + per-station hotspots ----
def split_mask(name):
    if name == "train":
        return (yr >= 2000) & (yr <= 2014)
    if name == "val":
        return (yr >= 2015) & (yr <= 2017)
    return (yr >= 2018) & (yr <= 2022)

print("\nEvent counts by split (DTF / FTD), per lead:")
for h in LEADS:
    lab = ev_labels[h]
    row = []
    for sp in ["train", "val", "test"]:
        m = split_mask(sp)[None, :] & np.isfinite(lab)
        row.append(f"{sp}: DTF={int(np.nansum((lab==1)&m))} FTD={int(np.nansum((lab==-1)&m))}")
    print(f"  h={h}  " + " | ".join(row))

print("\nPer-station event frequency (h=2, all years) - hotspot check:")
lab2 = ev_labels[2]
for s in range(S):
    v = np.isfinite(lab2[s])
    if v.sum() == 0:
        continue
    dtf = int(np.nansum(lab2[s] == 1)); ftd = int(np.nansum(lab2[s] == -1))
    print(f"  {stations[s]:22s} n={int(v.sum()):3d}  DTF={dtf:3d}  FTD={ftd:3d}  "
          f"event%={100*(dtf+ftd)/v.sum():4.1f}")

# ---- persist ----
np.savez_compressed(
    ART / "dfaa.npz",
    W=W, zP=zP, zSM=zSM, SPEI=SPEI,
    **{f"DFAA_h{h}": DFAA[h] for h in LEADS},
    **{f"WE_h{h}": W_E[h] for h in LEADS},
    **{f"WL_h{h}": W_L[h] for h in LEADS},
    **{f"lab_h{h}": ev_labels[h] for h in LEADS},
    stations=np.array(stations), yr=yr,
)
(ART / "dfaa_meta.json").write_text(json.dumps(meta, indent=2))
print(f"\nWrote {ART/'dfaa.npz'} and {ART/'dfaa_meta.json'}")
print("\nSTEP 2 OK.")
